In [ ]:
import numpy as np
import brian2 as b2
import matplotlib.pyplot as plt

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np
from numpy import exp


def simulate_bursting_neuron(i_ext=7.0 * b2.uA,
                             simulation_time=101 * b2.ms):

    """ 
    Simulation of bursting in Brian2.
    """

    # neuron parameters
    C = 1 * b2.ufarad
    g_na = 20 * b2.msiemens
    g_k = 10 * b2.msiemens
    g_l = 8 * b2.msiemens
    v_na = 60 * b2.mV
    v_k = -90 * b2.mV
    v_l = -80 * b2.mV
    tau_n = 0.15 * b2.ms
    tau_n_slow = 20 * b2.ms
    g_k_slow = 5 * b2.msiemens

    eqs = """
    n_slow_inf = 1./(1. + exp((-20.*mV-v)/(5.*mV))): 1
    n_inf = 1./(1. + exp((-25.*mV-v)/(5.*mV))): 1
    m_inf = 1./(1. + exp((-20.*mV-v)/(15.*mV))): 1
    
    membrane_Im = g_na * m_inf * (v_na - v) + g_k * n * (v_k - v) +
          g_k_slow * n_slow * (v_k-v) + g_l*(v_l-v) + i_ext : amp 
    
    dn/dt = (n_inf-n)/tau_n : 1
    dn_slow/dt = (n_slow_inf - n_slow) / tau_n_slow : 1
    dv/dt = membrane_Im / C : volt        
    """

    neurons = b2.NeuronGroup(1, eqs, method="rk4", dt=0.01*b2.ms)    
    neurons.v = -70.0*b2.mV
    neurons.n_slow = "n_slow_inf"
    neurons.n = "n_inf"
    state_mon = b2.StateMonitor(neurons, ["v"], record=True)
    net = b2.Network(neurons)
    net.add(state_mon)
    net.run(simulation_time)

    return state_mon

state_monitor = simulate_bursting_neuron(7.0 *b2.uA, 101 * b2.ms)

fig, ax = plt.subplots(1, figsize=(10, 3.5))
ax.plot(state_monitor.t / b2.ms, state_monitor.v[0] / b2.mV, lw=2, c="k")

ax.set_xlabel("t [ms]", fontsize=14)
ax.set_ylabel("v [mV]", fontsize=14)
ax.tick_params(labelsize=14)
# ax.legend(frameon=False)
ax.margins(x=0.0)
ax.set_yticks([-80,-50, 0])
plt.tight_layout()
plt.show();

In [ ]:
import numpy as np
import brian2 as b2
from numpy import exp
import matplotlib.pyplot as plt


def simulate_bursting_neurons(I_e=7.0*b2.uA,
                              simulation_time=101*b2.ms,
                              weight=0.25,
                              delay=1.*b2.ms,
                              g_syn=1.0*b2.msiemens):
    """ 
    Simulation of coupled bursting neurons in Brian2.
    """

    # neuron parameters
    C = 1 * b2.ufarad
    g_na = 20 * b2.msiemens
    g_k = 10 * b2.msiemens
    g_l = 8 * b2.msiemens
    v_na = 60 * b2.mV
    v_k = -90 * b2.mV
    v_l = -80 * b2.mV
    tau_n = 0.15 * b2.ms
    tau_n_slow = 20 * b2.ms
    g_k_slow = 5 * b2.msiemens

    tau_r = 0.2 * b2.ms
    tau_d = 3.0 * b2.ms

    eqs = """
    n_slow_inf = 1./(1. + exp((-20.*mV-v)/(5.*mV))): 1
    n_inf = 1./(1. + exp((-25.*mV-v)/(5.*mV))): 1
    m_inf = 1./(1. + exp((-20.*mV-v)/(15.*mV))): 1
    
    s_in: 1
    i_ext: amp
    
    membrane_Im = g_na * m_inf * (v_na - v) + g_k * n * (v_k - v) +
          g_k_slow * n_slow * (v_k-v) + g_l*(v_l-v) + i_ext +
          g_syn * s_in * (-v): amp 
    
    ds/dt = 0.5 * (1 + tanh(0.1*v/mV)) * (1-s)/tau_r - s/tau_d : 1
    dn/dt = (n_inf-n)/tau_n : 1
    dn_slow/dt = (n_slow_inf - n_slow) / tau_n_slow : 1
    dv/dt = membrane_Im / C : volt        
    """

    neurons = b2.NeuronGroup(2, eqs, 
                             method="rk4", 
                             threshold='v>-30*mV',
                             refractory='v>-30*mV',
                             dt=0.01*b2.ms)
    neurons.v = [-70.0, -65] * b2.mV
    neurons.n_slow = "n_slow_inf"
    neurons.i_ext = [I_e, 0*b2.uA]
    neurons.n = "n_inf"
    neurons.s_in = [0, 0]

    S = b2.Synapses(neurons,
                    neurons,
                    '''
                    w : 1
                    s_in_post = w*s_pre:1 (summed)
                    ''')
    S.connect(i=0, j=1)
    S.w[0, 1] = weight
#     S.delay = 'j*1*ms'

    state_mon = b2.StateMonitor(neurons, ["v"], record=[0, 1])
    net = b2.Network(neurons)
    net.add(state_mon)
    net.add(S)
    net.run(simulation_time)

    return state_mon


state_monitor = simulate_bursting_neurons(7.0 * b2.uA, 101 * b2.ms)

In [ ]:
fig, ax = plt.subplots(2, figsize=(12, 3.5))
ax[0].plot(state_monitor.t / b2.ms, state_monitor.v[0] / b2.mV, lw=2, c="r", alpha=0.5, label="neuron 0")
ax[1].plot(state_monitor.t / b2.ms, state_monitor.v[1] / b2.mV, lw=2, c="b", alpha=0.5, label='neuron 1')

for i in range(2):
    ax[i].set_xlabel("t [ms]")
    ax[i].set_ylabel("v [mV]")
    ax[i].legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.show();